In [18]:
import timm
import torch
import torch.nn as nn
import os
os.chdir('./../')

In [3]:
from src.models.sngp.sgnp_classification_layer import SNGP

In [4]:
sngp_classifier = SNGP(
                in_features=512,
                num_classes=10,
                kernel_scale_trainable=True,
                scale_random_features=True,
                normalize_input=False,
                covariance_momentum=0.999,
                return_dict=False,
            )

In [7]:
sngp_classifier.gp_classifier.reset_precision()

In [20]:
# Test the model with random input
model_name = "resnet50"
pretrained=True
cache_dir = 'timm_cache_dir'
in_chans = 3
num_classes = 8
backbone = timm.create_model(
            model_name,
            num_classes=num_classes,
            pretrained=pretrained,
            cache_dir=cache_dir)

batch_size = 2
in_chans = 3
height = width = 224
x = torch.randn(batch_size, in_chans, height, width)
out = backbone(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {out.shape}")

Input shape: torch.Size([2, 3, 224, 224])
Output shape: torch.Size([2, 8])


In [24]:
backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            cache_dir=cache_dir)

out = backbone.forward_features(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {out.shape}")

Input shape: torch.Size([2, 3, 224, 224])
Output shape: torch.Size([2, 2048, 7, 7])


In [23]:

backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,   # removes classifier head
            in_chans=in_chans,
            global_pool='',   # we control pooling
            cache_dir=cache_dir 
        )
out = backbone.forward(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {out.shape}")

out = backbone.forward_features(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {out.shape}")

Input shape: torch.Size([2, 3, 224, 224])
Output shape: torch.Size([2, 2048, 7, 7])
Input shape: torch.Size([2, 3, 224, 224])
Output shape: torch.Size([2, 2048, 7, 7])


Input shape: torch.Size([2, 3, 224, 224])
Output shape: torch.Size([2, 2048, 7, 7])


In [16]:
in_feats = getattr(backbone, "num_features", None)

In [ ]:

reduction_dim = 256
sngp_classifier = SNGP(
                in_features=reduction_dim,
                num_classes=num_classes,
                kernel_scale_trainable=True,
                scale_random_features=True,
                normalize_input=False,
                covariance_momentum=0.999,
                return_dict=False,
            ) # H -> H -> R -> D (R=128 D=1024) R < H, D >> R

In [27]:
class TimmSNGPClassifier(nn.Module):
    def __init__(self, 
                 model_name, 
                 num_classes, 
                 pretrained=True, 
                 in_chans=3,
                 reduction_dim=512):
        super().__init__()

        self.backbone = timm.create_model(
                            model_name,
                            pretrained=pretrained,
                            cache_dir=cache_dir)
        
        in_feats = getattr(self.backbone, "num_features", None)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.reduce_dim = nn.Linear(in_feats, reduction_dim)

        self.sngp_classifier = SNGP(
                in_features=reduction_dim,
                num_classes=num_classes,
                kernel_scale_trainable=True,
                scale_random_features=True,
                normalize_input=False,
                covariance_momentum=0.999,
                return_dict=False,
            )
        
    def forward(self, x):
        x = self.backbone.forward_features(x)
        x = self.pool(x).flatten(1)
        x = self.reduce_dim(x)
        return self.sngp_classifier(x)[0]

In [30]:
model_name = "resnet50"
num_classes = 8
pretrained=True 
in_chans=3
reduction_dim=512
sngp_model = TimmSNGPClassifier(model_name, 
                 num_classes, 
                 pretrained=True, 
                 in_chans=3,
                 reduction_dim=512)
x = torch.randn(batch_size, in_chans, height, width)
out = sngp_model(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {out.shape}")

Input shape: torch.Size([2, 3, 224, 224])
Output shape: torch.Size([2, 8])


In [8]:
nums = [1, 2]
res = []
def bt(ix, curr_state):
    if ix == len(nums):
        res.append(curr_state[:])
        return
    
    curr_state.append(nums[ix])
    bt(ix+1, curr_state)
    curr_state.pop()
    bt(ix+1, curr_state)

bt(0, [])

In [9]:
res

[[1, 2], [1], [2], []]

In [ ]:
# deal later
# batch_size = 2
# in_chans = 3
# height = width = 518
# x = torch.randn(batch_size, in_chans, height, width)
# out = backbone.forward_features(x)
# print(f"Input shape: {x.shape}")
# print(f"Output shape: {out.shape}")
# model_name = 'vit_large_patch14_dinov2.lvd142m' 
# pretrained=True
# cache_dir = 'timm_cache_dir'
# in_chans = 3
# num_classes = 8
# backbone = timm.create_model(
#             model_name,
#             pretrained=pretrained)